In [0]:
revenue_df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/sql_problems/default/my_volume/day04_daily_revenue.csv")

display(revenue_df)


In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, sum

# Define the window specification
windowSpec = Window.partitionBy(col("sales_rep")) \
                   .orderBy(col("transaction_date"))

# Apply the cumulative sum
result_df = (
    revenue_df
    .withColumn(
        "cumulative_sum", 
        sum(col("revenue")).over(windowSpec)
        )
)
result_df.show()

In [0]:
revenue_df.createOrReplaceTempView("revenue_table")


In [0]:
%sql
SELECT sales_rep, 
    transaction_date, 
    revenue,
    SUM(revenue) OVER(
        PARTITION BY sales_rep
        ORDER BY transaction_date
    ) as comm_sum
FROM revenue_table 


In [0]:
result_df.write.mode("overwrite").saveAsTable("day04_result_table").save("sql_problems.default/day04_result_table")

In [0]:
%sql
DESCRIBE EXTENDED day04_result_table